In [1]:
"""
================================================================================
MAIN.IPYNB - SOURCE-FREE OPEN-SET DOMAIN ADAPTATION PIPELINE
================================================================================

This notebook implements the complete pipeline for Source-Free Open-Set 
Domain Adaptation (SF-OSDA) on histopathology image datasets.

The pipeline consists of:
1. Dataset preparation (Kather16 and Kather19)
2. Source model training on labeled source data
3. Target model initialization from source model
4. Source-free adaptation on unlabeled target data
5. Open-set evaluation (known classification + unknown detection)

datasets:
- Kather16
- Kather19
"""

'\n================================================================================\nMAIN.IPYNB - SOURCE-FREE OPEN-SET DOMAIN ADAPTATION PIPELINE\n================================================================================\n\nThis notebook implements the complete pipeline for Source-Free Open-Set \nDomain Adaptation (SF-OSDA) on histopathology image datasets.\n\nThe pipeline consists of:\n1. Dataset preparation (Kather16 and Kather19)\n2. Source model training on labeled source data\n3. Target model initialization from source model\n4. Source-free adaptation on unlabeled target data\n5. Open-set evaluation (known classification + unknown detection)\n\ndatasets:\n- Kather16\n- Kather19\n'

In [2]:
# =============================================================================
# SECTION 1: IMPORTS AND SETUP
# =============================================================================

import torch
import torchvision
import shutil
from torch.utils.data import DataLoader
import os
import os.path as osp
from PIL import Image
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from typing import Tuple

from Data_loader import get_histopathology_datasets
from Data_loader import get_histopathology_transform
from split_kather import split_kather16, split_kather19
from utils_data import IndexedImageFolder
from Data_loader import osr_splits
from train_source import train_source_model, load_source_model
from Data_loader import load_source_dataset, load_target_dataset, get_histopathology_transform


In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)


Using device: cpu


In [4]:
# =============================================================================
# SECTION 2: DATASET PREPARATION
# =============================================================================
# After running this cell, kather19 and kather16 folders are created that include train, val and test folders.
# -----------------------------------------------------------------------------
# Split Kather datasets into train/val/test folders
# -----------------------------------------------------------------------------
# These functions organize the raw image files into structured folders
# for training, validation, and testing.

#split_kather16(r"C:\Users\USER\Mahdieh Nabavizadeh\Data\Kather 16\kather2016_image_tiles_5000\Kather_texture_2016_image_tiles_5000")
#split_kather19(r"C:\Users\USER\Mahdieh Nabavizadeh\Data\Kather 19\NCT CRC HE 100K\NCT-CRC-HE-100K")


# Current paths:
split_kather16(r"C:\Users\Asus\datasets\Kather 16\kather2016_image_tiles_5000\Kather_texture_2016_image_tiles_5000")
split_kather19(r"C:\Users\Asus\datasets\Kather 19\NCT CRC HE 100K\NCT-CRC-HE-100K")


Kather16 split complete (files copied).
Kather19 split complete (files copied).


In [5]:
# =============================================================================
# SECTION 3: SOURCE DATA LOADING AND CONFIGURATION
# =============================================================================
# -----------------------------------------------------------------------------
# Source dataset: Kather19 (used as labeled source)
# -----------------------------------------------------------------------------
# Split 1 defines which classes are "known" vs "unknown" for open-set setting
# Known classes: [8, (5, 7), 3, 6]
# This means classes 5 and 7 are merged into a single superclass

# Source data

dataset_name = "kather19"
split_idx = 1 # split 1

# Get the known classes for split 1
known_classes_s = osr_splits[dataset_name]['splits'][split_idx - 1]

# Example known classes
# known_classes_s = [8, (5, 7), 3, 6]

# Flatten the known_classes_s into a list for mapping
flattened_classes = []
for c in known_classes_s:
    if isinstance(c, tuple):
        flattened_classes.extend(c)
    else:
        flattened_classes.append(c)

# Create a label mapping: original_label -> new_index
# label_map = {}
# new_idx = 0
# for c in known_classes_s:
#     if isinstance(c, tuple):
#         for sub_c in c:
#             label_map[sub_c] = new_idx
#     else:
#         label_map[c] = new_idx
#     new_idx += 1
# print("Label map:", label_map)
# Output should be: {8:0, 5:1, 7:1, 3:2, 6:3}

print("Known classes in source dataset for split 1:", known_classes_s)

# Training the source model on source data (kather 19) - SUPERVISED 
# source data don't have unknown classes.
# -----------------------------------------------------------------------------
# Load source dataset (SUPERVISED - known classes only)
# -----------------------------------------------------------------------------
# Source data contains only known classes.

train_transform, test_transform = get_histopathology_transform(224)
source_dataset = load_source_dataset(
    root_dir=r"C:\Users\Asus\datasets\Kather 19\NCT CRC HE 100K\kather19",
    train_transform=train_transform,
    test_transform=test_transform,
    known_classes=known_classes_s, # we train on known classes from split 1
    unknown_classes=tuple([]),
    return_idx=True
)

# -----------------------------------------------------------------------------
# Source training configuration
# -----------------------------------------------------------------------------
class CFG:
    exp_name = "source_mobilenetv2_kather19"
    source_num_classes = len(known_classes_s)
    epochs = 5 # <-- change later <--- train on 20 epochs for kather19 
    lr = 1e-5
    weight_decay = 1e-5

cfg_s = CFG()

# -----------------------------------------------------------------------------
# Source DataLoaders
# -----------------------------------------------------------------------------
# Build source loaders
source_loaders = {
    "train": DataLoader(source_dataset['train'], batch_size=8, shuffle=True), # change batch_size to 64 ************
    "val":   DataLoader(source_dataset['val'], batch_size=8, shuffle=False), # change batch_size to 64 ************
    "test":  DataLoader(source_dataset['test'], batch_size=8, shuffle=False) # change batch_size to 64 ************
}

print("Number of training samples:", len(source_dataset['train']))
print("Number of validation samples:", len(source_dataset['val']))
print("Number of test samples:", len(source_dataset['test']))

print("Validation known samples:", len(source_dataset['val_known']))
print("Validation unknown samples:", len(source_dataset['val_unknown']))

print("Test known samples:", len(source_dataset['test_known']))
print("Test unknown samples:", len(source_dataset['test_unknown']))

print('Number of known classes:', len(known_classes_s))

print("labels in source training set:", set(source_dataset['train'].targets))


Known classes in source dataset for split 1: [8, (5, 7), 3, 6]
Training Known Dataset size: 1005
Number of training samples: 1005
Number of validation samples: 215
Number of test samples: 220
Validation known samples: 215
Validation unknown samples: 0
Test known samples: 220
Test unknown samples: 0
Number of known classes: 4
labels in source training set: {0, 1, 2, 3}


In [6]:
# =============================================================================
# SECTION 4: SOURCE MODEL TRAINING
# =============================================================================
# -----------------------------------------------------------------------------
# Train the source model on labeled source data
# -----------------------------------------------------------------------------

exp_dir, best_val, best_test = train_source_model(
    cfg_s=cfg_s, 
    datasets=source_loaders, 
    device=device
)

# -----------------------------------------------------------------------------
# Load training history
# -----------------------------------------------------------------------------
from pathlib import Path
history_path = Path(exp_dir) / "checkpoints" / "training_history.pth"
history = torch.load(history_path)

# Access epoch metrics
train_losses = history["train_loss"]
val_losses = history["val_loss"]
train_accs = history["train_acc"]
val_accs = history["val_acc"]



Epoch 1/5


Train Loss: 0.9869 | Train Acc: 0.6378


Val Loss:   0.5136 | Val Acc:   0.9070
Saved new best model (Val Acc: 0.9070)
Training history saved at: runs\source_mobilenetv2_kather19\checkpoints\training_history.pth

Epoch 2/5


Train Loss: 0.5383 | Train Acc: 0.8547


Val Loss:   0.2508 | Val Acc:   0.9581
Saved new best model (Val Acc: 0.9581)
Training history saved at: runs\source_mobilenetv2_kather19\checkpoints\training_history.pth

Epoch 3/5


...

Train Loss: 0.3071 | Train Acc: 0.9174


Val Loss:   0.1106 | Val Acc:   0.9767
Saved new best model (Val Acc: 0.9767)
Training history saved at: runs\source_mobilenetv2_kather19\checkpoints\training_history.pth

Epoch 5/5


Train Loss: 0.2677 | Train Acc: 0.9164


Val Loss:   0.1009 | Val Acc:   0.9767
Training history saved at: runs\source_mobilenetv2_kather19\checkpoints\training_history.pth


Best model test accuracy: 0.9864


In [7]:
# -----------------------------------------------------------------------------
# section 5 : Load the trained source model
# -----------------------------------------------------------------------------
source_model = load_source_model(exp_dir, known_classes=known_classes_s, device=device)

#print(f'number of known classes in source: {cfg_s.source_num_classes}')


Loaded source model from runs\source_mobilenetv2_kather19\checkpoints\best.pth with best val acc: 0.9767441860465116


In [8]:
# =============================================================================
# SECTION 6: TARGET DATA LOADING AND CONFIGURATION
# =============================================================================
# -----------------------------------------------------------------------------
# Target dataset: Kather16 (unlabeled)
# -----------------------------------------------------------------------------

# target data

from Data_loader import get_histopathology_datasets, load_target_dataset
from torch.utils.data import DataLoader

dataset_name = "kather16"
split_idx = 1 # split 1

# known classes for split 1
known_classes_split = osr_splits[dataset_name]['splits'][split_idx - 1]

known_classes_t = []
for c in known_classes_split:
    if isinstance(c, tuple):
        known_classes_t += list(c)
    else:
        known_classes_t.append(c)

# unknown classes
max_n_classes = osr_splits[dataset_name]["n_classes"]
unknown_classes_t = [x for x in range(max_n_classes) if x not in known_classes_t]

# TARGET DATASET IS UNLABELED ********
# -----------------------------------------------------------------------------
# Load target dataset (UNLABELED)
# -----------------------------------------------------------------------------
# Target data is unlabeled - we only use images, not labels
# However, we load the dataset with known/unknown splits for evaluation

# load unlabeled target dataset (Kather 16)
# transforms
train_transform, test_transform = get_histopathology_transform(224)
# load target dataset 
target_dataset = load_target_dataset(
    root_dir=r"C:\Users\Asus\datasets\Kather 16\kather2016_image_tiles_5000\kather16",
    train_transform=train_transform,
    test_transform=test_transform,
    known_classes=tuple(known_classes_t),
    unknown_classes=tuple(unknown_classes_t),
    return_idx=True)

# -----------------------------------------------------------------------------
# Target DataLoaders
# -----------------------------------------------------------------------------
# number of images in target training dataset
train_dataset = target_dataset['train']
print("Target training dataset size:", len(train_dataset))

# Known vs unknown counts in training set
train_known = target_dataset['train'].datasets[0] # first part of the IndexedConcatDataset
train_unknown = target_dataset['train'].datasets[1] # second part
print("Target training known dataset size:", len(train_known))
print("Target training unknown dataset size:", len(train_unknown))

# see test set
print("Target test dataset size:", len(target_dataset['test']))

target_loaders = {
    "train": DataLoader(target_dataset['train'], batch_size=8, shuffle=True), # change batch_size to 64 ************
    "test": DataLoader(target_dataset['test'], batch_size=8, shuffle=False), # change batch_size to 64 ************
    "test_known": DataLoader(target_dataset['test_known'], batch_size=8, shuffle=False), # change batch_size to 64 ************
    "test_unknown": DataLoader(target_dataset['test_unknown'], batch_size=8, shuffle=False) # change batch_size to 64 ************
}

n_classes = len(known_classes_t) + len(unknown_classes_t)
print("Total classes for target model:", n_classes)


Target training dataset size: 1552
Target training known dataset size: 776
Target training unknown dataset size: 776
Target test dataset size: 328
Total classes for target model: 8


In [9]:
# =============================================================================
# SECTION 7: TARGET MODEL INITIALIZATION
# =============================================================================

import math
import torch
import torch.optim as optim
from a1 import compute_cluster_entropy, compute_cluster_entropy_from_loader, ResidualBlock, TargetModel, squared_cdist, compute_q_ij, probe_layers
k = cfg_s.source_num_classes  # number of closed-set (known) classes
import copy

# -----------------------------------------------------------------------------
# Create target model from source model
# -----------------------------------------------------------------------------
# The target model shares the encoder with the source model but adds:
# 1. Residual block for bottom branch
# 2. Style module for augmentation
# 3. Cluster centers for target data
# 4. Class prototypes for distillation

target_model = TargetModel(
    encoder=copy.deepcopy(source_model.encoder),
    source_classifier=copy.deepcopy(source_model.fc),
    feature_dim=source_model.dim,
    num_source_classes=cfg_s.source_num_classes,
    use_style=True,
    style_eps=1e-5
).to(device)

# Freeze the classifier (source classifier is frozen during adaptation)
for p in target_model.classifier.parameters():
    p.requires_grad = False

# -----------------------------------------------------------------------------
# Initialize clustering and entropy threshold
# -----------------------------------------------------------------------------
# Initialize k-means clusters on target data
target_model.update_kmeans(target_loaders["train"], device=device, num_iters=40)

# Compute initial cluster entropy
E0 = compute_cluster_entropy_from_loader(target_model, target_loaders["train"], device)

# Calibrate target variance for variance anchoring
target_model.calibrate_target_variance(target_loaders["train"], device)

# Store cluster entropies
maxH = math.log(target_model.num_clusters + 1e-12)
E0_clean = E0.clone()
E0_clean[~torch.isfinite(E0_clean)] = maxH
target_model.saved_cluster_entropies.copy_(E0_clean)


qmax mean/min/max: 0.47861626744270325 0.21161003410816193 0.8896345496177673
cluster counts: [166.0, 158.0, 257.0, 240.0, 161.0, 201.0, 162.0, 207.0]


tensor([1.6027, 1.3477, 1.6253, 0.8657, 1.5270, 1.2596, 1.4876, 1.6330])

In [10]:
# =============================================================================
# SECTION 8: ADAPTATION CONFIGURATION
# =============================================================================
# -----------------------------------------------------------------------------
# Verify encoder initialization (should match source model)
# -----------------------------------------------------------------------------

for p1, p2 in zip(source_model.encoder.parameters(),
                  target_model.encoder.parameters()):
    print(torch.equal(p1, p2))
    break # should print true
    

True


In [11]:
# Initialize the target model
import math
from a1 import compute_cluster_entropy, compute_cluster_entropy_from_loader, ResidualBlock, TargetModel, squared_cdist, compute_q_ij
k = cfg_s.source_num_classes  # number of closed-set (known) classes

# -----------------------------------------------------------------------------
# Adaptation hyperparameters
# -----------------------------------------------------------------------------
num_epochs_adapt = 8

# Optimizer: update encoder and residual ************************************************************************style??
optimizer = optim.Adam([{"params": target_model.encoder.parameters(), "lr": 1e-4, "weight_decay": 1e-4},
        {"params": target_model.residual.parameters(), "lr": 1e-4, "weight_decay": 1e-4},])

# Add learning rate scheduler
from torch.optim.lr_scheduler import StepLR
scheduler = StepLR(optimizer, step_size=4, gamma=0.5)  # Decay learning rate by 0.5 every 4 epochs

# -----------------------------------------------------------------------------
# Loss weight scheduling (curriculum learning)
# -----------------------------------------------------------------------------
def get_lambdas(epoch, last_epoch_loss):
    """
    Stable curriculum for source-free open-set adaptation.
    Gradual ramp-up + adaptive backoff if loss increases.
    
    The schedule progressively introduces:
    - Phase 1: CE + Anchor + tiny style (representation stabilization)
    - Phase 2: Gentle prototype introduction
    - Phase 3: Controlled distillation
    - Phase 4: Full objective (conservative)
    
    Adaptive backoff reduces unstable terms if loss increases.
    """
    # Phase 1: Representation Stabilization 
    # Only CE + Anchor + tiny style
    if epoch < 2:
        lmb = dict(
            proto=0.0,
            distill=0.0,
            style=0.005,
            catkl=0.0005,
            anchor=0.5,
            var_anchor=0.02,)
    # Phase 2: Gentle Prototype Introduction
    elif epoch < 4:
        lmb = dict(
            proto=0.05,
            distill=0.0,
            style=0.01,
            catkl=0.001,
            anchor=0.3,
            var_anchor=0.05,
        )
    # Phase 3: Controlled Distillation
    elif epoch < 6:
        lmb = dict(
            proto=0.1,
            distill=0.01,
            style=0.01,
            catkl=0.002,
            anchor=0.15,
            var_anchor=0.05,
        )
    # Phase 4: Full Objective (but still conservative) 
    else:
        lmb = dict(
            proto=0.15,
            distill=0.02,
            style=0.01,
            catkl=0.003,
            anchor=0.1,
            var_anchor=0.05,
        )
    # ADAPTIVE BACKOFF IF LOSS INCREASES
    if not math.isinf(last_epoch_loss):
        if hasattr(get_lambdas, "prev_loss"):
            if last_epoch_loss > get_lambdas.prev_loss:
                # reduce unstable terms
                lmb["proto"] *= 0.7
                lmb["distill"] *= 0.7
                lmb["catkl"] *= 0.5
                lmb["style"] *= 0.8

        get_lambdas.prev_loss = last_epoch_loss
    else:
        get_lambdas.prev_loss = last_epoch_loss

    return lmb

# =============================================================================
# SECTION 9: LAYER PROBING (Find optimal freezing boundary)
# =============================================================================
# -----------------------------------------------------------------------------
# Inspect available layer names
# -----------------------------------------------------------------------------
# inspect available submodule names first
print([n for n, _ in target_model.encoder.named_modules() if n])

candidate_layers = ["features.14", "features.16", "features.17", "features.18"]  

# -----------------------------------------------------------------------------
# Probe layers for optimal freezing boundary
# -----------------------------------------------------------------------------
# This finds the layer where features best separate known from unknown **************************************************************
results = probe_layers(
    target_model.encoder,
    candidate_layers,
    target_loaders["test_known"],
    target_loaders["test_unknown"],
    device,
    num_clusters=target_model.num_clusters,
)

def _combined_score(r):
    """Combine silhouette and AUROC for layer ranking."""
    s = r["silhouette"]
    a = r["auroc_known_vs_unknown"]
    if s != s:  # NaN check
        s = -1.0
    if a != a:
        a = 0.5
    return s + a

# -----------------------------------------------------------------------------
# Freeze encoder up to selected layer
# -----------------------------------------------------------------------------
def freeze_encoder_up_to(encoder, layer_name, verbose=True):
    """
    Freezes encoder parameters up to and including `layer_name`
    (dotted path, e.g. 'features.14'); everything strictly after it
    stays trainable. Handles one level of nesting into an indexed
    Sequential container (e.g. MobileNetV2's `features`).
    """
    parts = layer_name.split(".")
    top_target = parts[0]
    sub_target = parts[1] if len(parts) > 1 else None

    reached_target = False
    for top_name, top_module in encoder.named_children():
        if reached_target:
            for p in top_module.parameters():
                p.requires_grad = True
            if verbose:
                print(f"Trainable: {top_name}")
            continue

        if top_name != top_target:
            for p in top_module.parameters():
                p.requires_grad = False
            if verbose:
                print(f"Frozen: {top_name}")
            continue

        if sub_target is None:
            for p in top_module.parameters():
                p.requires_grad = False
            if verbose:
                print(f"Frozen: {top_name} (entire block)")
            reached_target = True
            continue

        inner_reached = False
        for sub_name, sub_module in top_module.named_children():
            if inner_reached:
                for p in sub_module.parameters():
                    p.requires_grad = True
                if verbose:
                    print(f"Trainable: {top_name}.{sub_name}")
                continue
            for p in sub_module.parameters():
                p.requires_grad = False
            if verbose:
                print(f"Frozen: {top_name}.{sub_name}")
            if sub_name == sub_target:
                inner_reached = True
        reached_target = True

# Select best layer based on combined score
best_layer = max(results.items(), key=lambda kv: _combined_score(kv[1]))[0]
print(f"[Layer probe] Selected freeze boundary: {best_layer}")

# Freeze layers up to the selected boundary
freeze_encoder_up_to(target_model.encoder, best_layer)

# Print probe results
for name, r in results.items():
    print(f"{name}: silhouette={r['silhouette']:.4f}  AUROC(known-vs-unknown)={r['auroc_known_vs_unknown']:.4f}")
    
entropy_history = []
import torch.nn as nn

# =============================================================================
# SECTION 10: FREEZE BATCH NORM (Stable adaptation)
# =============================================================================
def freeze_bn(m):
    """Freeze batch norm statistics and parameters."""
    if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
        m.eval()
        for p in m.parameters():
            p.requires_grad = False

target_model.apply(freeze_bn)

# =============================================================================
# SECTION 11: ADAPTATION TRAINING LOOP
# =============================================================================
last_epoch_loss = float('inf')

for epoch in range(num_epochs_adapt):
    # -------------------------------------------------------------------------
    # Get loss weights for this epoch
    # -------------------------------------------------------------------------
    lmb = get_lambdas(epoch, last_epoch_loss)
    target_model.train()

    total_loss_epoch = 0.0
    style_loss_epoch = 0.0
    proto_loss_epoch = 0.0
    distill_loss_epoch = 0.0
    cat_kl_loss_epoch = 0.0
    cluster_entropy_loss_epoch = 0.0
    num_batches = 0

    # -------------------------------------------------------------------------
    # Training loop over target data
    # -------------------------------------------------------------------------
    for it, batch in enumerate(target_loaders["train"]):
        x_t = batch[0].to(device)

        # Forward pass
        outputs = target_model(x_t, return_all=True)
        
        w = outputs["w_final"]
        cons_w = min(0.3, 0.05 + 0.05 * epoch)
        # cluster_entropy_loss = outputs["cluster_entropy_loss"]
        style_loss = outputs["style_loss"]
        proto_loss = outputs["proto_loss"]
        distill_loss = outputs["distill_loss"]
        cat_kl_loss = outputs["cat_kl_loss"]
        anchor_loss = outputs["anchor_loss"]
        loss = (
            outputs["loss_known_ce"]
            + lmb["proto"] * outputs["proto_loss"]
            + lmb["distill"] * outputs["distill_loss"] 
            # + 0.5 * outputs["logit_consistency_loss"]
            # + min(0.5, 0.1 + 0.1 * epoch) * outputs["logit_consistency_loss"]s
            # + cons_w * outputs["logit_consistency_loss"]
            + 0.0 * outputs["logit_consistency_loss"] # We don’t need it. Prototype distill already aligns branches ******
            + lmb["anchor"] * outputs["anchor_loss"]
            + lmb["style"] * outputs["style_loss"]
            + lmb["catkl"] * outputs["cat_kl_loss"]
            + lmb["var_anchor"] * outputs["variance_anchor_loss"]
            + 0.1 * outputs["loss_balance"]
        )

        # Skip invalid batches
        if torch.any(torch.isnan(loss)) or torch.any(torch.isinf(loss)):
            print(f"Loss contains NaN or Inf. Skipping batch.")
            continue

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(target_model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss_epoch += float(loss.item())
        style_loss_epoch += float(style_loss.item())
        proto_loss_epoch += float(proto_loss.item())
        distill_loss_epoch += float(distill_loss.item())
        cat_kl_loss_epoch += float(cat_kl_loss.item())
        # cluster_entropy_loss_epoch += float(cluster_entropy_loss.item())
        num_batches += 1

        # Debug output every 50 iterations
        if it % 50 == 0:
            known_ratio = (outputs["w_final"] >= 0.5).float().mean().item()
            unknown_ratio = (outputs["w_final"] < 0.5).float().mean().item()
            cs = outputs["conf_st"].mean().item()
            print(f" [it {it:04d}] "
                f"known_ratio={known_ratio:.3f} "
                f"unknown_ratio={unknown_ratio:.3f} "
                f"mean(conf_st)={cs:.3f}")

    # Step learning rate scheduler
    scheduler.step()

    # -------------------------------------------------------------------------
    # Compute average losses
    # -------------------------------------------------------------------------
    avg_total_loss = total_loss_epoch / max(num_batches, 1)
    avg_style_loss = style_loss_epoch / max(num_batches, 1)
    avg_proto_loss = proto_loss_epoch / max(num_batches, 1)
    avg_distill_loss = distill_loss_epoch / max(num_batches, 1)
    # avg_cluster_entropy_loss = cluster_entropy_loss_epoch / max(num_batches, 1)
    avg_cat_kl_loss = cat_kl_loss_epoch / max(num_batches, 1)
    last_epoch_loss = avg_total_loss

    # CLUSTER UPDATE (EVERY 2 EPOCHS, CONSISTENT FEATURES)
    # -------------------------------------------------------------------------
    # Update clusters every 2 epochs
    # -------------------------------------------------------------------------
    target_model.eval()
    if epoch % 2 == 0:
        print(f"[Epoch {epoch:02d}] Updating k-means clusters")
        target_model.update_kmeans(target_loaders["train"],
            device=device,
            num_iters=100)
        E_k_epoch = compute_cluster_entropy_from_loader(
            target_model,
            target_loaders["train"],
            device)
        
        # Calibrate entropy threshold
        from a1 import calibrate_cluster_entropy_threshold
        tau, _ = calibrate_cluster_entropy_threshold(target_model, target_loaders["train"], device)
        print(f"[Epoch {epoch:02d}] Data-driven entropy threshold (Otsu) = {tau:.4f}")
    else:
        print(f"[Epoch {epoch:02d}] Reusing previous clusters")
        E_k_epoch = target_model.saved_cluster_entropies.clone()

    # -------------------------------------------------------------------------
    # Update class prototypes (balanced)
    # -------------------------------------------------------------------------
    print(f"[Epoch {epoch:02d}] Updating class prototypes (balanced)")
    target_model.eval()
    with torch.no_grad():
        feats = []
        preds = []
        for batch in target_loaders["train"]:
            x = batch[0].to(device)
            out = target_model(x, return_all=True)
            conf = out["conf_st"]
            pred = out["logits_st"].argmax(dim=1)
            # FIX: LOWERED FROM 0.85 TO 0.6
            mask = conf >= 0.6
            if mask.any():
                feats.append(out["f_t"][mask].detach())
                preds.append(pred[mask].detach())
            
        if len(feats) > 0:
            feats_all = torch.cat(feats)
            preds_all = torch.cat(preds)
            # Count class distribution
            counts = torch.bincount(preds_all, minlength=k)
            print("Pseudo-label counts:", counts.tolist())
            total_selected = counts.sum().item()
            dominant_class_ratio = counts.max().item() / max(total_selected, 1)
            if dominant_class_ratio > 0.7:  
                print(">>> Skipping prototype update (dominant class detected)")
            else:
                for c in range(k):
                    idx = preds_all == c
                    if idx.any():
                        mean_feat = F.normalize(
                            feats_all[idx].mean(dim=0),
                            dim=0
                        )
                        target_model.class_prototypes[c].copy_(mean_feat)

        # FALLBACK - If no prototypes updated, use k-means centers
        if len(feats) == 0 or all((target_model.class_prototypes.norm(dim=1) <= 1e-6).tolist()):
            print(">>> No prototypes found, using k-means centers as fallback")
            # Use first k cluster centers as prototypes
            for c in range(min(k, target_model.num_clusters)):
                target_model.class_prototypes[c] = F.normalize(
                    target_model.cluster_centers[c], 
                    dim=0, 
                    eps=1e-12
                )

        # -------------------------------------------------------------------------
        # Store entropy history
        # -------------------------------------------------------------------------
        finite = E_k_epoch[torch.isfinite(E_k_epoch)]
        if finite.numel() > 0:
            mean_E = finite.mean()
            std_E  = finite.std()
            tau = mean_E + 0.5 * std_E
        else:
            mean_E = torch.tensor(float("nan"), device=E_k_epoch.device)
            tau = torch.tensor(float("inf"), device=E_k_epoch.device)
        
        known_mask = E_k_epoch < tau

        # store finite entropies
        maxH = math.log(target_model.num_clusters + 1e-12)
        E_k_epoch_clean = E_k_epoch.clone()
        E_k_epoch_clean[~torch.isfinite(E_k_epoch_clean)] = maxH
        target_model.saved_cluster_entropies.copy_(E_k_epoch_clean)

        entropy_history.append({
            "epoch": epoch,
            "E_k": E_k_epoch.detach().cpu().numpy(),
            "mean_E": mean_E,
            "losses": {"total": avg_total_loss,
                "style": avg_style_loss,
                "proto": avg_proto_loss,
                "distill": avg_distill_loss,
                "cat_kl": avg_cat_kl_loss,
                "logit_consistency_loss": float(outputs["logit_consistency_loss"].item()),}
        })

        print(f"[Epoch {epoch:02d}] mean(E_k) = {mean_E:.6f}")
        print(f"[Epoch {epoch:02d}] Losses -> "
              f"Total: {avg_total_loss:.6f}, "
              f"Style: {avg_style_loss:.6f}, "
              f"Proto: {avg_proto_loss:.6f}, "
              f"Distill: {avg_distill_loss:.6f}, "
              # f"ClusterEnt: {avg_cluster_entropy_loss:.6f}, "
              f"CatKL: {avg_cat_kl_loss:.6f}",
              f"logit_consistency_loss: {outputs['logit_consistency_loss'].item():.6f}")


['features', 'features.0', 'features.0.0', 'features.0.1', 'features.0.2', 'features.1', 'features.1.conv', 'features.1.conv.0', 'features.1.conv.0.0', 'features.1.conv.0.1', 'features.1.conv.0.2', 'features.1.conv.1', 'features.1.conv.2', 'features.2', 'features.2.conv', 'features.2.conv.0', 'features.2.conv.0.0', 'features.2.conv.0.1', 'features.2.conv.0.2', 'features.2.conv.1', 'features.2.conv.1.0', 'features.2.conv.1.1', 'features.2.conv.1.2', 'features.2.conv.2', 'features.2.conv.3', 'features.3', 'features.3.conv', 'features.3.conv.0', 'features.3.conv.0.0', 'features.3.conv.0.1', 'features.3.conv.0.2', 'features.3.conv.1', 'features.3.conv.1.0', 'features.3.conv.1.1', 'features.3.conv.1.2', 'features.3.conv.2', 'features.3.conv.3', 'features.4', 'features.4.conv', 'features.4.conv.0', 'features.4.conv.0.0', 'features.4.conv.0.1', 'features.4.conv.0.2', 'features.4.conv.1', 'features.4.conv.1.0', 'features.4.conv.1.1', 'features.4.conv.1.2', 'features.4.conv.2', 'features.4.conv

In [12]:
# =============================================================================
# SECTION 12: OPEN-SET RECOGNITION CALIBRATION
# =============================================================================
# -----------------------------------------------------------------------------
# Reload model if needed
# -----------------------------------------------------------------------------

# import importlib
# import a1

# sd = target_model.state_dict()

# # target_model_new = a1.TargetModel(
# #     encoder=target_model.encoder,    # USE ADAPTED ENCODER
# #     source_classifier=target_model.classifier,
# #     feature_dim=source_model.dim,
# #     num_source_classes=cfg_s.source_num_classes,
# #     use_style=True,
# #     style_eps=1e-5).to(device)
# target_model_new = TargetModel(
#     encoder=copy.deepcopy(source_model.encoder),
#     source_classifier=source_model.classifier,
#     feature_dim=source_model.dim,
#     num_source_classes=cfg_s.source_num_classes,
#     use_style=True,
#     style_eps=1e-5
# ).to(device)
# target_model_new.load_state_dict(sd, strict=False)
# target_model = target_model_new



In [13]:
# -----------------------------------------------------------------------------
# Verify test set labels
# -----------------------------------------------------------------------------

test_known = target_dataset['test'].datasets[0]
test_unknown = target_dataset['test'].datasets[1]

known_labels_test = set(test_known.targets)
unknown_labels_test = set(test_unknown.targets)
print("labels in target test known set:", known_labels_test)


labels in target test known set: {0, 1, 2, 3}


In [14]:
# -----------------------------------------------------------------------------
# Calibrate OSR thresholds
# -----------------------------------------------------------------------------

from a1 import (
    calibrate_entropy_threshold,
    compute_cluster_entropy,
    compute_cluster_entropy_from_loader,
    TargetModel,
    ResidualBlock,)

# Calibrate OSR thresholds 
target_model.eval()

# -----------------------------------------------------------------------------
# Re-calibrate entropy threshold for OSR
# -----------------------------------------------------------------------------
# 1) Entropy threshold 
tau_entropy, entropies = calibrate_entropy_threshold(target_model,
    target_loaders["train"],
    device,
    q=0.80,
    return_all=True)

# 2) Prototype distance threshold
target_model.calibrate_proto_threshold(target_loaders["train"],device)

print("Calibrated proto_dist_threshold:",
      float(target_model.proto_dist_threshold.item()))


Calibrated proto_dist_threshold: 0.05738675594329834


In [15]:
# =============================================================================
# SECTION 13: FORWARD PASS TEST
# =============================================================================

# Test time
target_model.eval()
k = cfg_s.source_num_classes  # unknown index = k
# Entropy OSR calibration
from a1 import calibrate_entropy_threshold

tau_entropy, entropies = calibrate_entropy_threshold(
    target_model,
    target_loaders["train"],
    device,
    q=0.80,
    return_all=True)

print(f"[OSR] calibrated entropy threshold = {tau_entropy:.4f}")
print("Entropy stats (train):",
      "min", entropies.min().item(),
      "mean", entropies.mean().item(),
      "max", entropies.max().item())


[OSR] calibrated entropy threshold = 0.5015
Entropy stats (train): min 4.963691679904514e-08 mean 0.20824876427650452 max 1.3818118572235107


In [16]:
target_model.eval()
with torch.no_grad():
    x = next(iter(target_loaders["test_known"]))[0].to(device)
    out = target_model(x, return_all=True)
    print(out.keys())


dict_keys(['f_st', 'f_t', 'logits_st', 'logits_t', 'probs_t', 'loss_known_ce', 'proto_loss', 'distill_loss', 'cat_kl_loss', 'logit_consistency_loss', 'anchor_loss', 'style_loss', 'conf_st', 'energy_t', 'energy_st', 'w_final', 'loss_balance', 'variance_anchor_loss'])


In [17]:
# =============================================================================
# SECTION 14: OPEN-SET EVALUATION
# =============================================================================
# -----------------------------------------------------------------------------
# OOD Score Functions
# -----------------------------------------------------------------------------

# Distill-SODA MLS
def mls(logits: torch.Tensor) -> torch.Tensor:
    """
    Maximum Logit Score (MLS)
    logits: [N, K]
    returns: [N]
    """
    return logits.max(dim=1).values
# NEGATIVE ENTROPY 
def neg_entropy(logits: torch.Tensor) -> torch.Tensor:
    probs = torch.softmax(logits, dim=1)
    entropy = -(probs * torch.log(probs + 1e-12)).sum(dim=1)
    return -entropy
# AUROC 
# Quality of known/unknown separation
def auroc(scores: torch.Tensor, is_known: torch.Tensor):
    """
    AUROC without sklearn/scipy/numpy
    """
    scores = scores.detach().cpu()
    labels = is_known.detach().cpu().to(torch.int)
    # sort by score descending
    order = torch.argsort(scores, descending=True)
    labels = labels[order]
    P = labels.sum().item()
    N = (labels == 0).sum().item()
    if P == 0 or N == 0:
        return float("nan"), None, None, None, None
    tps = torch.cumsum(labels, dim=0)
    fps = torch.cumsum(1 - labels, dim=0)
    tpr = tps.float() / P
    fpr = fps.float() / N
    auroc_value = torch.trapz(tpr, fpr).item()
    return auroc_value, fpr, tpr, None, None
# OOD Evaluation (Distill-SODA style: MLS + AUROC)
target_model.eval()
all_logits = []
all_is_known = []
with torch.no_grad():
    #Known samples 
    for batch in target_loaders["test_known"]:
        x = batch[0].to(device)
        out = target_model(x, return_all=True)
        logits = out["logits_st"]   # TOP branch classifier logits
        # logits = out["logits_t"]
        all_logits.append(logits.cpu())
        all_is_known.append(torch.ones(x.size(0), dtype=torch.bool))
    # Unknown samples 
    for batch in target_loaders["test_unknown"]:
        x = batch[0].to(device)
        out = target_model(x, return_all=True)
        logits = out["logits_st"]
        # logits = out["logits_t"]
        all_logits.append(logits.cpu())
        all_is_known.append(torch.zeros(x.size(0), dtype=torch.bool))
all_logits = torch.cat(all_logits, dim=0)
mask_id = torch.cat(all_is_known, dim=0)
# MLS AUROC
mls_scores = mls(all_logits)
auroc_mls, _, _, _, _ = auroc(mls_scores, mask_id)
print(f"[OOD] AUROC (MLS) = {auroc_mls:.4f}")
# Negative Entropy AUROC
negent_scores = neg_entropy(all_logits)
auroc_negent, _, _, _, _ = auroc(negent_scores, mask_id)
print(f"[OOD] AUROC (NegEntropy) = {auroc_negent:.4f}")
# OOD SCORE USING w_final (KNOWN-CONFIDENCE)
all_w = []
with torch.no_grad():
    # Known samples (label = 1)
    for batch in target_loaders["test_known"]:
        x = batch[0].to(device)
        out = target_model(x, return_all=True)
        all_w.append(out["w_final"].cpu())
    # Unknown samples (label = 0)
    for batch in target_loaders["test_unknown"]:
        x = batch[0].to(device)
        out = target_model(x, return_all=True)
        all_w.append(out["w_final"].cpu())
all_w = torch.cat(all_w, dim=0)
# Higher score must mean "more likely known"
# w_final already represents known confidence
known_scores = all_w
auroc_, fpr, tpr, _, _ = auroc(known_scores, mask_id)
print(f"[OOD] AUROC (w_final) = {auroc_:.4f}")


[OOD] AUROC (MLS) = 0.6678
[OOD] AUROC (NegEntropy) = 0.7117
[OOD] AUROC (w_final) = 0.7111


In [18]:
# =============================================================================
# SECTION 15: CLOSED-SET ACCURACY EVALUATION
# =============================================================================
# -----------------------------------------------------------------------------
# Accuracy on known classes
# -----------------------------------------------------------------------------
correct = 0
total = 0
with torch.no_grad():
    for batch in target_loaders["test_known"]:
        x, y = batch[0].to(device), batch[1].to(device)
        out = target_model(x, return_all=True)
        # preds = out["logits_t"].argmax(dim=1)
        preds = out["logits_st"].argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.numel()
acc_known = correct / total
print(f"[Closed-set] Accuracy = {acc_known:.4f}")


[Closed-set] Accuracy = 0.4573


In [19]:
from collections import Counter

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in target_loaders["test_known"]:
        x, y = batch[0].to(device), batch[1].to(device)
        out = target_model(x, return_all=True)
        preds = out["logits_st"].argmax(dim=1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(y.cpu().tolist())
# see if The model predicts only 2 classes (1 and 2)
print("Prediction distribution:", Counter(all_preds))
print("True distribution:", Counter(all_labels))


Prediction distribution: Counter({2: 122, 1: 34, 3: 8})
True distribution: Counter({0: 41, 1: 41, 2: 41, 3: 41})


In [20]:
# # source_model = load_pretrained_source_model()

# # =============================================================================
# # SECTION 16: SOURCE MODEL TEST ACCURACY (REFERENCE)
# # =============================================================================
# # -----------------------------------------------------------------------------
# # Evaluate source model on source test data ******(for comparison)******
# # -----------------------------------------------------------------------------

# source_model.eval()
# correct = 0
# total = 0
# with torch.no_grad():
#     for batch in source_loaders["test"]:
#         x, y = batch[0].to(device), batch[1].to(device)
#         logits = source_model(x)
#         preds = logits.argmax(dim=1)
#         correct += (preds == y).sum().item()
#         total += y.numel()
# print("Source test accuracy on source test data:", correct/total)
